<a href="https://colab.research.google.com/github/sayam-h069/Compiler_Design/blob/main/CD_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1

In [15]:
!apt-get update -qq
!apt-get install -y flex gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
flex is already the newest version (2.6.4-8.2build1).
gcc is already the newest version (4:13.2.0-7ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [23]:
%%writefile employee.l
%{
#include <stdio.h>

int valid = 0;
int invalid = 0;
%}

%%

^[A-Z]{2}_?[0-9]{3}[a-z]*\n {
    printf("Valid Employee ID   : %.*s\n", yyleng - 1, yytext);
    valid++;
}

^[^\n]*\n {
    printf("Invalid Employee ID : %.*s\n", yyleng - 1, yytext);
    invalid++;
}

[ \t]+ ;

%%

int yywrap()
{
    return 1;
}

int main()
{
    printf("Enter Employee IDs (one per line):\n");
    yylex();

    printf("\n-------------------------\n");
    printf("Valid IDs   : %d\n", valid);
    printf("Invalid IDs : %d\n", invalid);

    return 0;
}

Overwriting employee.l


In [24]:
!flex employee.l
!gcc lex.yy.c -o employee
!printf "HR_101\nIT205dev\nCS_999temp\nH_101\nHR101A\nHR_10\n" | ./employee

Enter Employee IDs (one per line):
Valid Employee ID   : HR_101
Valid Employee ID   : IT205dev
Valid Employee ID   : CS_999temp
Invalid Employee ID : H_101
Invalid Employee ID : HR101A
Invalid Employee ID : HR_10

-------------------------
Valid IDs   : 3
Invalid IDs : 3


In [25]:
!printf "CS_123\nAB456test\nA_123\nAB_12\nAB_123TEST\nXY999hello\n" | ./employee

Enter Employee IDs (one per line):
Valid Employee ID   : CS_123
Valid Employee ID   : AB456test
Invalid Employee ID : A_123
Invalid Employee ID : AB_12
Invalid Employee ID : AB_123TEST
Valid Employee ID   : XY999hello

-------------------------
Valid IDs   : 3
Invalid IDs : 3


2

In [19]:
%%writefile source.c
int a = 10;
float b = a + 20;
if(a < b)
{
    return a;
}

Writing source.c


In [20]:
!cat source.c

int a = 10;
float b = a + 20;
if(a < b)
{
    return a;
}


In [21]:
%%writefile analyzer.l
%{
#include <stdio.h>

int keywords = 0;
int identifiers = 0;
int numbers = 0;
int operators = 0;
int special_symbols = 0;
int lines = 0;
%}

%%

"int"|"float"|"if"|"else"|"while"|"return" {
    keywords++;
}

[a-zA-Z][a-zA-Z0-9]* {
    identifiers++;
}

[0-9]+ {
    numbers++;
}

[+\-*/=] {
    operators++;
}

[;,(){}] {
    special_symbols++;
}

\n {
    lines++;
}

[ \t]+ {
    /* Ignore spaces and tabs */
}

. {
    /* Ignore other characters */
}

%%

int yywrap()
{
    return 1;
}

int main()
{
    FILE *fp;

    fp = fopen("source.c", "r");

    if (fp == NULL)
    {
        printf("Error: Cannot open source.c\n");
        return 1;
    }

    yyin = fp;

    yylex();

    fclose(fp);

    printf("\n----- Lexical Statistics -----\n");
    printf("Keywords       : %d\n", keywords);
    printf("Identifiers     : %d\n", identifiers);
    printf("Numbers         : %d\n", numbers);
    printf("Operators       : %d\n", operators);
    printf("Special Symbols : %d\n", special_symbols);
    printf("Lines           : %d\n", lines);

    return 0;
}

Writing analyzer.l


In [22]:
!flex analyzer.l
!gcc lex.yy.c -o analyzer
!./analyzer


----- Lexical Statistics -----
Keywords       : 4
Identifiers     : 6
Numbers         : 2
Operators       : 3
Special Symbols : 7
Lines           : 6


3

In [26]:
%%writefile marks.txt
101 CS 85
102 MA 35
10 CS 90
103 EE 120

Writing marks.txt


In [27]:
!cat marks.txt

101 CS 85
102 MA 35
10 CS 90
103 EE 120


In [28]:
%%writefile marks.l
%{
#include <stdio.h>

int total = 0;
int valid = 0;
int invalid = 0;
int passed = 0;
int failed = 0;

char roll[20];
char subject[20];
int marks;
%}

%%

^[0-9]{3}[ \t]+[A-Z]{2}[ \t]+(100|[0-9]{1,2})[ \t]*\n {
    total++;
    valid++;

    sscanf(yytext, "%s %s %d", roll, subject, &marks);

    if (marks >= 40) {
        passed++;
        printf("Valid Record - PASS : %s %s %d\n",
               roll, subject, marks);
    }
    else {
        failed++;
        printf("Valid Record - FAIL : %s %s %d\n",
               roll, subject, marks);
    }
}

^[0-9]{3}[ \t]+[A-Z]{2}[ \t]+[0-9]+[ \t]*\n {
    total++;
    invalid++;

    sscanf(yytext, "%s %s %d", roll, subject, &marks);

    printf("Invalid Record (Marks out of range) : %s %s %d\n",
           roll, subject, marks);
}

^[^\n]*\n {
    total++;
    invalid++;

    printf("Invalid Record : %.*s\n",
           yyleng - 1, yytext);
}

[ \t]+ ;

\n ;

%%

int yywrap()
{
    return 1;
}

int main()
{
    FILE *fp;

    fp = fopen("marks.txt", "r");

    if (fp == NULL)
    {
        printf("Error: Cannot open marks.txt\n");
        return 1;
    }

    yyin = fp;

    yylex();

    fclose(fp);

    printf("\n------ Student Marks Report ------\n");
    printf("Total Records  : %d\n", total);
    printf("Valid Records  : %d\n", valid);
    printf("Invalid Records: %d\n", invalid);
    printf("Passed         : %d\n", passed);
    printf("Failed         : %d\n", failed);

    return 0;
}

Writing marks.l


In [29]:
!flex marks.l
!gcc lex.yy.c -o marks

In [30]:
!./marks

Valid Record - PASS : 101 CS 85
Valid Record - FAIL : 102 MA 35
Invalid Record : 10 CS 90
Invalid Record (Marks out of range) : 103 EE 120

------ Student Marks Report ------
Total Records  : 4
Valid Records  : 2
Invalid Records: 2
Passed         : 1
Failed         : 1
